# Text-to-SQL with Guardrails

Text-to-SQL turns a business question into SQL using an LLM. It matters because it can make analytics more accessible to non-SQL users, but it also introduces risk: the model can generate incorrect, overly expensive, or even dangerous SQL. In this walkthrough you will see the full pattern: schema-aware prompting, SQL generation, validation guardrails, and safe execution against DuckDB.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.setup_db import setup_database, get_schema

conn = setup_database("data/sales.csv")
schema = get_schema(conn)
print(schema)
conn.execute("SELECT * FROM sales LIMIT 5").fetch_df()


In [ ]:
from src.generate_sql import build_system_prompt

system_prompt = build_system_prompt(schema)
print(system_prompt)


In [ ]:
from src.llm import get_client, generate_sql

client = get_client()  # Requires OPENAI_API_KEY in your environment
questions = [
    "What are the top 5 products by revenue?",
    "How many completed orders came from each region?",
    "What is the average order value for each category?",
]

for question in questions:
    raw_sql = generate_sql(question, schema, client)
    print(f"Question: {question}")
    print(raw_sql)
    print("-" * 80)


In [ ]:
from src.validate import run_guardrails

dangerous_sql = "DROP TABLE sales"
run_guardrails(dangerous_sql, conn)


In [ ]:
from src.execute import run_safe_query

sql = "SELECT product, ROUND(SUM(total_amount), 2) AS revenue FROM sales GROUP BY product ORDER BY revenue DESC LIMIT 5"
result = run_safe_query(sql, conn)
print(result["results"])
conn.execute(sql).fetch_df()


In [ ]:
# Ambiguous questions are common in production.
# Example: does "best region" mean highest revenue, most orders, or highest average order value?
# The model will make a reasonable assumption based on the prompt, but this is where product design matters.
ambiguous_question = "Which region is performing best?"
ambiguous_sql = generate_sql(ambiguous_question, schema, client)
print(ambiguous_sql)
run_safe_query(ambiguous_sql, conn)


## Adapting to production

- Replace DuckDB with your warehouse connection (Databricks SQL, Snowflake, BigQuery, Postgres).
- Expand guardrails with table allow-lists, query complexity checks, and human approval for sensitive datasets.
- Log prompts, generated SQL, validation failures, and user feedback for continuous improvement.
- Add semantic layers or metric definitions so the LLM uses business-approved logic instead of guessing.
- Consider caching, async APIs, and observability for higher-volume applications.
